# 1. ADAS preprocessing

This notebook is part of the reproducible data-preparation pipeline used by the downstream modelling notebooks.


## 1.1. Connect Google Drive

connect this notebook to my Google Drive so that I can access the downloaded ADNI tables and save all intermediate and processed ADAS outputs persistently.

In [ ]:
from google.colab import drive

# Mount Google Drive so the notebook can access the ADNI data folders.
drive.mount("/content/drive")

## 1.2. Load the raw ADAS table

load the complete ADAS assessment table from the cognitive and functional assessments folder. At this stage, preserve the raw data exactly as downloaded and only verify that the correct file has been found and loaded successfully.

In [ ]:
from pathlib import Path

import pandas as pd

# Define the path to the downloaded ADAS table.
ADAS_PATH = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/raw/"
    "Cognitive and functional assessments/"
    "All_Subjects_ADAS_11Jul2026.csv"
)

# Stop with a clear error if the path or filename is incorrect.
if not ADAS_PATH.exists():
    raise FileNotFoundError(
        f"The ADAS file was not found at:\n{ADAS_PATH}\n\n"
        "Check the folder names, capitalisation, and filename in Google Drive."
    )

# Load the raw ADAS table without preprocessing.
adas_raw = pd.read_csv(ADAS_PATH, low_memory=False)

print(f"Loaded file: {ADAS_PATH.name}")
print(f"Number of rows: {adas_raw.shape[0]:,}")
print(f"Number of columns: {adas_raw.shape[1]:,}")

display(adas_raw.head())

## 1.3. Inspect the ADAS table structure

examine the column names, data types, missing-value counts, and a small sample of the raw ADAS table. This will help me identify the participant identifiers, visit variables, assessment dates, ADAS item scores, total scores, and administrative fields before deciding what should be retained or removed.

In [ ]:
# Display the overall dimensions of the raw ADAS table.
print(f"Rows: {adas_raw.shape[0]:,}")
print(f"Columns: {adas_raw.shape[1]:,}")

# Print every column name with its data type and one non-missing example.
column_summary = []

for column in adas_raw.columns:
    non_missing_values = adas_raw[column].dropna()

    example_value = (
        non_missing_values.iloc[0]
        if not non_missing_values.empty
        else None
    )

    column_summary.append(
        {
            "column": column,
            "dtype": str(adas_raw[column].dtype),
            "non_missing": int(adas_raw[column].notna().sum()),
            "missing": int(adas_raw[column].isna().sum()),
            "missing_percent": round(
                adas_raw[column].isna().mean() * 100,
                2
            ),
            "sample_value": example_value,
        }
    )

adas_column_summary = pd.DataFrame(column_summary)

# Show all rows and avoid truncating long text values.
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

display(adas_column_summary)

## 1.4. Load the ADNI data dictionary

load the ADNI data dictionary from the raw cohort and source-of-truth folder. The dictionary contains the official descriptions of variables, coded response values, units, and source tables. first inspect its structure before matching it to the columns in the ADAS table.

In [ ]:
from pathlib import Path

import pandas as pd

# Define the path to the ADNI data dictionary.
DATADIC_PATH = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging/raw/"
    "Cohort, dates and source-of-truth tables/"
    "DATADIC_11Jul2026.csv"
)

# Confirm that the dictionary file exists before loading it.
if not DATADIC_PATH.exists():
    raise FileNotFoundError(
        f"The data dictionary was not found at:\n{DATADIC_PATH}"
    )

# Load the data dictionary without altering its contents.
datadic_raw = pd.read_csv(DATADIC_PATH, low_memory=False)

print(f"Loaded file: {DATADIC_PATH.name}")
print(f"Number of rows: {datadic_raw.shape[0]:,}")
print(f"Number of columns: {datadic_raw.shape[1]:,}")

# Display all dictionary column names and one example value from each.
datadic_structure = pd.DataFrame(
    {
        "column": datadic_raw.columns,
        "dtype": [
            str(datadic_raw[column].dtype)
            for column in datadic_raw.columns
        ],
        "sample_value": [
            (
                datadic_raw[column].dropna().iloc[0]
                if datadic_raw[column].notna().any()
                else None
            )
            for column in datadic_raw.columns
        ],
    }
)

display(datadic_structure)
display(datadic_raw.head())

## 1.5. Match the ADAS table columns to the official data dictionary

filter the ADNI data dictionary to entries belonging to the ADAS table and retain only variables that are present in the downloaded ADAS dataset.

The same variable may have multiple dictionary rows because ADNI phases and case-report-form versions sometimes use different wording, coding, or harmonisation rules. therefore preserve those rows initially so that phase-specific differences are visible before any preprocessing decisions are made.

In [ ]:
# Standardise the dictionary field names and ADAS column names for matching.
datadic_raw["TBLNAME_CLEAN"] = (
    datadic_raw["TBLNAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

datadic_raw["FLDNAME_CLEAN"] = (
    datadic_raw["FLDNAME"]
    .astype("string")
    .str.strip()
    .str.upper()
)

adas_columns_upper = {
    str(column).strip().upper()
    for column in adas_raw.columns
}

# Keep dictionary rows that belong to the ADAS table and correspond
# to columns present in the downloaded ADAS dataset.
adas_dictionary = datadic_raw.loc[
    (datadic_raw["TBLNAME_CLEAN"] == "ADAS")
    & (datadic_raw["FLDNAME_CLEAN"].isin(adas_columns_upper))
].copy()

# Retain the original dictionary fields in a readable order.
adas_dictionary = adas_dictionary[
    [
        "PHASE",
        "CRFNAME",
        "TBLNAME",
        "FLDNAME",
        "TEXT",
        "TYPE",
        "LENGTH",
        "DD_CRF_VERSION",
        "CODE",
        "UNITS",
        "STATUS",
        "CODE_CHANGES",
        "MAPPING_NOTES",
    ]
].sort_values(
    by=["FLDNAME", "PHASE", "DD_CRF_VERSION"],
    na_position="last"
).reset_index(drop=True)

print(f"ADAS columns in downloaded table: {len(adas_raw.columns):,}")
print(
    "ADAS columns with at least one dictionary match: "
    f"{adas_dictionary['FLDNAME'].nunique():,}"
)
print(f"Dictionary rows retained: {len(adas_dictionary):,}")

# Identify downloaded columns that do not have an ADAS dictionary match.
matched_fields = set(
    adas_dictionary["FLDNAME"]
    .astype(str)
    .str.strip()
    .str.upper()
)

unmatched_adas_columns = [
    column
    for column in adas_raw.columns
    if str(column).strip().upper() not in matched_fields
]

print("\nColumns without an ADAS dictionary match:")
print(unmatched_adas_columns)

display(adas_dictionary)

## 1.6. Interpretation of the ADAS Data Dictionary Output

The ADNI data dictionary contained 575 rows corresponding to 122 of the 124 columns in the downloaded ADAS table. The only columns without a direct dictionary match were `PHASE` and `UPDATE_STAMP`. This indicates that almost all variables in the consolidated ADAS dataset have official definitions and coding information available across the different ADNI phases.

The large number of dictionary rows does not mean that the ADAS table contains 575 distinct variables. Most variables appear several times because the dictionary provides separate definitions for ADNI1, ADNIGO, ADNI2, ADNI3, and ADNI4. In many cases, the variable meaning is consistent across phases, while the original coding or wording differs slightly.

The dictionary confirms that the downloaded ADAS table is already substantially harmonised across ADNI phases. Earlier variables were mapped into a common naming and coding structure, as indicated by fields such as `STATUS`, `CODE_CHANGES`, and `MAPPING_NOTES`. For example, several earlier-phase task codes were shifted or remapped so that the same values have consistent interpretations in later phases.

The table contains several broad categories of variables:

1. **Participant and visit identifiers**

   These include `PTID`, `RID`, `VISCODE`, `VISCODE2`, and `VISDATE`. They identify the participant and the visit at which the assessment was completed. `VISDATE` represents the assessment date where available, otherwise a matching date from the ADNI registry.

2. **Assessment completion and quality-control fields**

   `DONE` indicates whether the assessment was completed, while `NDREASON` records why it was not completed. `HAS_QC_ERROR` identifies ADNI4 records with unresolved quality-control problems. `SOURCE` indicates whether information was collected during an in-person visit or remotely.

3. **Raw task-response variables**

   Variables such as `Q1TR1`, `Q2TASK`, `Q5NAME1`, and `Q8WORD1` contain detailed responses to individual ADAS tasks. Some text variables contain colon-separated codes representing several incorrect, recalled, or completed items. These provide very fine-grained information but are more difficult to harmonise and use directly in the planned neural-network model.

4. **Task-completion variables**

   Variables ending in `UNABLE`, such as `Q1UNABLE` and `Q13UNABLE`, describe whether an individual task was conducted or not. Typical codes indicate refusal, physical inability, cognitive inability, or another reason for non-completion. These values should not be treated as ordinary cognitive scores.

5. **Harmonised component scores**

   `Q1SCORE` through `Q13SCORE` are calculated ADAS score components. They correspond to cognitive domains including word recall, commands, constructional praxis, delayed recall, naming, ideational praxis, orientation, word recognition, remembering instructions, comprehension, word finding, language, and number cancellation.

   Higher values indicate poorer performance or greater cognitive impairment.

6. **Total ADAS scores**

   `TOTSCORE` is the classic ADAS-Cog 11 score, with a range from 0 to 70. It excludes delayed word recall and number cancellation.

   `TOTAL13` is the expanded ADAS-Cog 13 score, with a range from 0 to 85. It includes delayed word recall and number cancellation.

   In both cases, a higher score indicates worse cognitive performance.

7. **Phase-specific variables**

   Some variables are only available in later ADNI phases. For example, `Q5SCORE_CUE`, `LANGUAGE_CODE`, `DD_CRF_VERSION_LABEL`, and `HAS_QC_ERROR` have high missingness because they were not collected consistently across all phases. Their missing values are therefore often structural rather than accidental.

8. **Administrative variables**

   `ID`, `SITEID`, `USERDATE`, `USERDATE2`, and `UPDATE_STAMP` relate to database records, sites, or data-entry timestamps. They do not describe participant cognition and should not be used as predictive model features.



## 1.7. Select the ADAS variables needed for preprocessing

The raw ADAS table contains detailed responses for individual words, drawings, commands, reminders, and test items. These fields are useful for reconstructing the assessment but are unnecessarily granular for the planned multimodal model.

retain participant and visit identifiers, assessment-completion information, the harmonised ADAS component scores, and the ADAS-11 and ADAS-13 total scores. Raw task responses and administrative database fields will not be carried into the working modelling table.

In [ ]:
# Define identifiers needed to match the assessment to participants and visits.
identifier_columns = [
    "PHASE",
    "PTID",
    "RID",
    "VISCODE",
    "VISCODE2",
    "VISDATE",
]

# Retain fields needed to determine whether the assessment was completed
# and whether an ADNI4 record has a quality-control error.
qc_columns = [
    "DONE",
    "NDREASON",
    "SOURCE",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
]

# Define the harmonised ADAS component scores.
component_score_columns = [
    "Q1SCORE",
    "Q2SCORE",
    "Q3SCORE",
    "Q4SCORE",
    "Q5SCORE",
    "Q5SCORE_CUE",
    "Q6SCORE",
    "Q7SCORE",
    "Q8SCORE",
    "Q9SCORE",
    "Q10SCORE",
    "Q11SCORE",
    "Q12SCORE",
    "Q13SCORE",
]

# Retain both total-score versions during QC.
total_score_columns = [
    "TOTSCORE",
    "TOTAL13",
]

# Select only columns that are actually present in the downloaded table.
requested_columns = (
    identifier_columns
    + qc_columns
    + component_score_columns
    + total_score_columns
)

available_columns = [
    column
    for column in requested_columns
    if column in adas_raw.columns
]

missing_requested_columns = [
    column
    for column in requested_columns
    if column not in adas_raw.columns
]

adas_working = adas_raw[available_columns].copy()

print(f"Raw ADAS columns: {adas_raw.shape[1]:,}")
print(f"Working ADAS columns: {adas_working.shape[1]:,}")
print(f"Working ADAS rows: {adas_working.shape[0]:,}")

print("\nRequested columns not found:")
print(missing_requested_columns)

display(adas_working.head())

## 1.8. Examine ADAS coverage across ADNI phases

inspect how many ADAS records and participants come from each ADNI phase. also calculate phase-specific missingness for the assessment-completion, language, source, quality-control, component-score, and total-score variables.

This is important because some administrative and quality-control variables were introduced only in later ADNI phases. Their missing values should not be interpreted as failed assessments without first considering the phase in which each record was collected.

In [ ]:
# Count assessment records and unique participants in each ADNI phase.
phase_summary = (
    adas_working
    .groupby("PHASE", dropna=False)
    .agg(
        assessment_rows=("RID", "size"),
        unique_participants=("RID", "nunique"),
        earliest_visit_date=("VISDATE", "min"),
        latest_visit_date=("VISDATE", "max"),
    )
    .reset_index()
)

print("ADAS records by ADNI phase:")
display(phase_summary)

# Define fields whose missingness may vary substantially by phase.
coverage_columns = [
    "DONE",
    "NDREASON",
    "SOURCE",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
    "Q1SCORE",
    "Q2SCORE",
    "Q3SCORE",
    "Q4SCORE",
    "Q5SCORE",
    "Q5SCORE_CUE",
    "Q6SCORE",
    "Q7SCORE",
    "Q8SCORE",
    "Q9SCORE",
    "Q10SCORE",
    "Q11SCORE",
    "Q12SCORE",
    "Q13SCORE",
    "TOTSCORE",
    "TOTAL13",
]

# Calculate the percentage of missing values for each field within each phase.
phase_missingness = (
    adas_working
    .groupby("PHASE")[coverage_columns]
    .apply(lambda group: group.isna().mean().mul(100).round(2))
)

print("\nPercentage missing within each ADNI phase:")
display(phase_missingness)

## 1.9. Convert the ADAS visit date to datetime

The `VISDATE` column was loaded as a generic object column rather than a true date. convert it to Pandas datetime format so that earliest and latest visit dates can be calculated correctly. Invalid or missing date values will be converted to `NaT` rather than causing an error.

In [ ]:
# Convert the visit date to a proper datetime variable.
adas_working["VISDATE"] = pd.to_datetime(
    adas_working["VISDATE"],
    errors="coerce"
)

print("VISDATE data type:")
print(adas_working["VISDATE"].dtype)

print("\nNumber of missing or invalid VISDATE values:")
print(adas_working["VISDATE"].isna().sum())

## 1.10. Summarise ADAS completion and quality-control fields

convert the observed values of the ADAS completion and quality-control variables into readable summary tables. Each table will show the original value, its interpretation from the ADNI data dictionary, the number of records, and the percentage of the full ADAS dataset.

Missing values will be labelled explicitly as either not collected or not applicable, rather than being displayed as `NaN`.

In [ ]:
# Define readable labels using the ADNI data dictionary.
qc_value_labels = {
    "DONE": {
        0.0: "No",
        1.0: "Yes",
    },
    "NDREASON": {
        1.0: "Unable for cognitive reasons",
        2.0: "Unable for other reasons",
        3.0: "Participant or study partner refused",
        7.0: "Site error",
        8.0: "Other reason",
        9.0: "Unknown",
    },
    "SOURCE": {
        1.0: "Participant visit",
        2.0: "Telephone or video call",
    },
    "LANGUAGE_CODE": {
        "e": "English",
        "s": "Spanish",
    },
    "HAS_QC_ERROR": {
        0.0: "No unresolved QC error",
        1.0: "Has unresolved QC error",
    },
}

qc_value_columns = [
    "DONE",
    "NDREASON",
    "SOURCE",
    "LANGUAGE_CODE",
    "HAS_QC_ERROR",
]

qc_summary_tables = {}

for column in qc_value_columns:
    # Count all observed and missing values.
    counts = (
        adas_working[column]
        .value_counts(dropna=False)
        .rename_axis("value")
        .reset_index(name="record_count")
    )

    # Add readable interpretations for each coded value.
    counts["interpretation"] = counts["value"].map(
        qc_value_labels[column]
    )

    # Give missing values an explicit description.
    counts.loc[
        counts["value"].isna(),
        "interpretation"
    ] = "Not collected or not applicable"

    # Calculate percentages relative to the entire ADAS table.
    counts["percent_of_records"] = (
        counts["record_count"]
        .div(len(adas_working))
        .mul(100)
        .round(2)
    )

    # Reorder the columns for readability.
    counts = counts[
        [
            "value",
            "interpretation",
            "record_count",
            "percent_of_records",
        ]
    ]

    qc_summary_tables[column] = counts

    print(f"\n{column}")
    display(counts)

## 1.11. Summarise ADAS total-score availability

create a readable table showing how many records contain the ADAS-Cog 11 and ADAS-Cog 13 total scores, together with their missing counts and coverage percentages.

In [ ]:
score_availability = pd.DataFrame(
    {
        "measure": [
            "TOTSCORE",
            "TOTAL13",
        ],
        "description": [
            "ADAS-Cog 11 total score",
            "ADAS-Cog 13 total score",
        ],
        "available_rows": [
            adas_working["TOTSCORE"].notna().sum(),
            adas_working["TOTAL13"].notna().sum(),
        ],
        "missing_rows": [
            adas_working["TOTSCORE"].isna().sum(),
            adas_working["TOTAL13"].isna().sum(),
        ],
        "available_percent": [
            round(
                adas_working["TOTSCORE"].notna().mean() * 100,
                2
            ),
            round(
                adas_working["TOTAL13"].notna().mean() * 100,
                2
            ),
        ],
    }
)

display(score_availability)

## 1.12. Overview of ADAS completion, quality control, and score availability

The ADAS table contains 13,098 assessment records. The completion and quality-control variables are highly phase-dependent, so missing values in these fields should not automatically be interpreted as invalid assessments.

The `DONE` variable was available for 4,117 records. Among these, 4,003 assessments were marked as completed and 114 were marked as not completed. The remaining 8,981 records had no `DONE` value because this field was not collected in the earlier ADNI phases. Therefore, older records should not be excluded simply because `DONE` is missing.

The `NDREASON` variable was populated only for the 114 records where an assessment was not completed. The most common reason was an unspecified “other reason” with 68 records. Smaller numbers were attributed to cognitive inability, refusal, non-cognitive inability, site error, or an unknown reason. Its 99.13% missingness is therefore expected because the field is only relevant when `DONE = 0`.

The `SOURCE` variable was recorded for later-phase assessments. Among the available records, 2,167 assessments were completed during an in-person participant visit and only 12 were completed by telephone or video call. This indicates that nearly all assessments with recorded source information were administered in person.

The `LANGUAGE_CODE` variable was available only in ADNI4. Of the 1,478 records with language information, 1,458 were administered in English and 20 in Spanish. Because language was not recorded in earlier phases, missing language values should be treated as structural missingness rather than unknown language.

The `HAS_QC_ERROR` field was also restricted to ADNI4. Of the 1,478 ADNI4 records, 1,476 had no unresolved quality-control error and only 2 records were flagged with an unresolved QC error. These two records should be examined and excluded from the cleaned dataset if the flag remains unresolved. :contentReference[oaicite:0]{index=0}

Coverage of the principal ADAS total scores was very high. `TOTSCORE`, representing the ADAS-Cog 11 total score, was available for 12,884 records, corresponding to 98.37% of the table. `TOTAL13`, representing the ADAS-Cog 13 total score, was available for 12,780 records, corresponding to 97.57% coverage.

Overall, the ADAS modality has strong usable coverage. The main preprocessing rule should therefore be phase-aware: retain earlier-phase records when their score values are present, exclude explicitly incomplete later-phase assessments, exclude unresolved QC-error records, and avoid treating structurally absent administrative fields as ordinary missing data.

## 1.13. Create ADAS validity and quality-control flags

create transparent row-level flags describing whether each ADAS assessment is complete, free from unresolved quality-control errors, within the valid score ranges, and sufficiently identified for longitudinal matching.

not remove any records in this step. The flags will allow me to review the effect of each exclusion rule before constructing the cleaned ADAS dataset.

In [ ]:
# Create a copy so that the current working table remains unchanged.
adas_qc = adas_working.copy()

# An assessment is explicitly incomplete only when DONE is recorded as 0.
# Missing DONE values occur in earlier ADNI phases and are not treated as failures.
adas_qc["flag_explicitly_not_done"] = adas_qc["DONE"].eq(0)

# An unresolved quality-control problem is present only when the ADNI4 flag equals 1.
# Missing values in earlier phases are treated as not applicable.
adas_qc["flag_unresolved_qc_error"] = adas_qc["HAS_QC_ERROR"].eq(1)

# Identify records that do not contain either principal ADAS total score.
adas_qc["flag_missing_both_totals"] = (
    adas_qc["TOTSCORE"].isna()
    & adas_qc["TOTAL13"].isna()
)

# Check the official valid ranges of the two total scores.
adas_qc["flag_totscore_out_of_range"] = (
    adas_qc["TOTSCORE"].notna()
    & ~adas_qc["TOTSCORE"].between(0, 70)
)

adas_qc["flag_total13_out_of_range"] = (
    adas_qc["TOTAL13"].notna()
    & ~adas_qc["TOTAL13"].between(0, 85)
)

# Check whether essential participant and visit identifiers are absent.
adas_qc["flag_missing_rid"] = adas_qc["RID"].isna()
adas_qc["flag_missing_visit_date"] = adas_qc["VISDATE"].isna()

# Combine the main exclusion conditions into one provisional flag.
adas_qc["flag_provisional_exclusion"] = (
    adas_qc["flag_explicitly_not_done"]
    | adas_qc["flag_unresolved_qc_error"]
    | adas_qc["flag_missing_both_totals"]
    | adas_qc["flag_totscore_out_of_range"]
    | adas_qc["flag_total13_out_of_range"]
    | adas_qc["flag_missing_rid"]
    | adas_qc["flag_missing_visit_date"]
)

# Summarise the number and percentage of rows affected by each flag.
qc_flag_columns = [
    "flag_explicitly_not_done",
    "flag_unresolved_qc_error",
    "flag_missing_both_totals",
    "flag_totscore_out_of_range",
    "flag_total13_out_of_range",
    "flag_missing_rid",
    "flag_missing_visit_date",
    "flag_provisional_exclusion",
]

qc_flag_summary = pd.DataFrame(
    {
        "qc_flag": qc_flag_columns,
        "affected_rows": [
            int(adas_qc[column].sum())
            for column in qc_flag_columns
        ],
        "affected_percent": [
            round(adas_qc[column].mean() * 100, 2)
            for column in qc_flag_columns
        ],
    }
)

display(qc_flag_summary)

## 1.14. Inspect overlap between ADAS exclusion reasons

The provisional quality-control rules identify 217 unique records for exclusion, but several records may satisfy more than one rule. inspect the combinations of exclusion flags to understand why each record is being removed.

This ensures that incomplete assessments, missing scores, unresolved quality-control errors, and absent visit dates are handled transparently before creating the cleaned ADAS table.

In [ ]:
# Count how many exclusion conditions apply to each record.
individual_exclusion_flags = [
    "flag_explicitly_not_done",
    "flag_unresolved_qc_error",
    "flag_missing_both_totals",
    "flag_totscore_out_of_range",
    "flag_total13_out_of_range",
    "flag_missing_rid",
    "flag_missing_visit_date",
]

adas_qc["number_of_exclusion_reasons"] = (
    adas_qc[individual_exclusion_flags]
    .sum(axis=1)
)

# Create a readable description of the exclusion reason combination.
def describe_exclusion_reasons(row):
    reasons = []

    if row["flag_explicitly_not_done"]:
        reasons.append("Assessment explicitly not done")

    if row["flag_unresolved_qc_error"]:
        reasons.append("Unresolved QC error")

    if row["flag_missing_both_totals"]:
        reasons.append("Both total scores missing")

    if row["flag_totscore_out_of_range"]:
        reasons.append("TOTSCORE outside valid range")

    if row["flag_total13_out_of_range"]:
        reasons.append("TOTAL13 outside valid range")

    if row["flag_missing_rid"]:
        reasons.append("RID missing")

    if row["flag_missing_visit_date"]:
        reasons.append("Visit date missing")

    return " + ".join(reasons) if reasons else "No exclusion reason"


adas_qc["exclusion_reason"] = adas_qc.apply(
    describe_exclusion_reasons,
    axis=1
)

# Summarise only records provisionally marked for exclusion.
exclusion_overlap_summary = (
    adas_qc.loc[adas_qc["flag_provisional_exclusion"]]
    .groupby(
        [
            "exclusion_reason",
            "number_of_exclusion_reasons",
        ],
        dropna=False
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        by=["record_count", "exclusion_reason"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

exclusion_overlap_summary["percent_of_excluded_records"] = (
    exclusion_overlap_summary["record_count"]
    .div(adas_qc["flag_provisional_exclusion"].sum())
    .mul(100)
    .round(2)
)

display(exclusion_overlap_summary)

## 1.15. Create the cleaned longitudinal ADAS table

separate the valid ADAS assessments from the records that failed the quality-control rules.

The cleaned table will retain all valid longitudinal assessments for later baseline alignment. The excluded records will be saved separately with their exclusion reasons so that the preprocessing remains transparent and reproducible.

In [ ]:
# Separate valid and excluded ADAS assessment records.
adas_clean_longitudinal = (
    adas_qc.loc[~adas_qc["flag_provisional_exclusion"]]
    .copy()
    .reset_index(drop=True)
)

adas_excluded = (
    adas_qc.loc[adas_qc["flag_provisional_exclusion"]]
    .copy()
    .reset_index(drop=True)
)

print(f"Original ADAS rows: {len(adas_qc):,}")
print(f"Clean longitudinal ADAS rows: {len(adas_clean_longitudinal):,}")
print(f"Excluded ADAS rows: {len(adas_excluded):,}")
print(
    "Retained percentage: "
    f"{len(adas_clean_longitudinal) / len(adas_qc) * 100:.2f}%"
)

# Confirm that no retained records still meet an exclusion condition.
remaining_flagged_rows = int(
    adas_clean_longitudinal["flag_provisional_exclusion"].sum()
)

print(
    "\nFlagged rows remaining in cleaned table: "
    f"{remaining_flagged_rows}"
)

display(
    adas_clean_longitudinal[
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "TOTSCORE",
            "TOTAL13",
        ]
    ].head()
)

## 1.16. Check for duplicate ADAS assessments

inspect whether any participant has more than one valid ADAS record assigned to the same visit code or visit date.

This check is necessary before baseline alignment because duplicate assessment records may represent repeated entries, corrections, different ADNI phases, or genuinely separate assessments. identify them without removing anything yet.

In [ ]:
# Count exact duplicate rows across the retained working variables.
exact_duplicate_count = int(
    adas_clean_longitudinal.duplicated().sum()
)

# Identify repeated participant and translated visit-code combinations.
duplicate_visitcode_mask = (
    adas_clean_longitudinal
    .duplicated(
        subset=["RID", "VISCODE2"],
        keep=False
    )
)

duplicate_visitcode_records = (
    adas_clean_longitudinal.loc[
        duplicate_visitcode_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "TOTSCORE",
            "TOTAL13",
        ]
    ]
    .sort_values(
        by=["RID", "VISCODE2", "VISDATE", "PHASE"]
    )
    .reset_index(drop=True)
)

# Identify repeated participant and assessment-date combinations.
duplicate_date_mask = (
    adas_clean_longitudinal
    .duplicated(
        subset=["RID", "VISDATE"],
        keep=False
    )
)

duplicate_date_records = (
    adas_clean_longitudinal.loc[
        duplicate_date_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "TOTSCORE",
            "TOTAL13",
        ]
    ]
    .sort_values(
        by=["RID", "VISDATE", "VISCODE2", "PHASE"]
    )
    .reset_index(drop=True)
)

duplicate_summary = pd.DataFrame(
    {
        "duplicate_check": [
            "Exact duplicate rows",
            "Rows in repeated RID + VISCODE2 combinations",
            "Unique repeated RID + VISCODE2 combinations",
            "Rows in repeated RID + VISDATE combinations",
            "Unique repeated RID + VISDATE combinations",
        ],
        "count": [
            exact_duplicate_count,
            int(duplicate_visitcode_mask.sum()),
            int(
                duplicate_visitcode_records[
                    ["RID", "VISCODE2"]
                ].drop_duplicates().shape[0]
            ),
            int(duplicate_date_mask.sum()),
            int(
                duplicate_date_records[
                    ["RID", "VISDATE"]
                ].drop_duplicates().shape[0]
            ),
        ],
    }
)

display(duplicate_summary)

## 1.17. Inspect same-day ADAS assessments

I found two participant-date combinations where the same participant has more than one ADAS record on the same calendar date. These are not exact duplicates and they do not share the same translated visit code.

inspect the four records in detail before deciding whether they represent valid screening and baseline assessments, duplicate data entry, or another visit-coding issue.

In [ ]:
# Display all ADAS records involved in repeated RID + VISDATE combinations.
same_day_adas_records = (
    adas_clean_longitudinal.loc[
        duplicate_date_mask,
        [
            "PHASE",
            "PTID",
            "RID",
            "VISCODE",
            "VISCODE2",
            "VISDATE",
            "DONE",
            "HAS_QC_ERROR",
            "TOTSCORE",
            "TOTAL13",
        ]
    ]
    .sort_values(
        by=["RID", "VISDATE", "VISCODE2"]
    )
    .reset_index(drop=True)
)

display(same_day_adas_records)

## 1.18. Exclude ambiguous same-day ADAS assessments

Two participants have two different ADAS visits recorded on the same calendar date. The paired records have different visit codes and different ADAS scores, so they cannot be treated as exact duplicates.

Because the ADAS table alone does not provide enough evidence to determine which visit date or score is correct, exclude all four ambiguous records. They will be retained in the exclusion log with a specific reason so that this decision remains auditable.

In [ ]:
# Identify the four records belonging to repeated RID + VISDATE combinations.
ambiguous_same_day_mask = (
    adas_clean_longitudinal
    .duplicated(
        subset=["RID", "VISDATE"],
        keep=False
    )
)

# Save the ambiguous records before removing them.
adas_same_day_excluded = (
    adas_clean_longitudinal.loc[ambiguous_same_day_mask]
    .copy()
)

adas_same_day_excluded["exclusion_reason"] = (
    "Ambiguous same-day records with different visit codes and scores"
)

# Remove the four ambiguous records from the cleaned longitudinal table.
adas_clean_longitudinal = (
    adas_clean_longitudinal.loc[~ambiguous_same_day_mask]
    .copy()
    .reset_index(drop=True)
)

# Add the four records to the existing exclusion log.
adas_excluded = pd.concat(
    [
        adas_excluded,
        adas_same_day_excluded,
    ],
    ignore_index=True,
    sort=False,
)

print(
    "Ambiguous same-day records excluded: "
    f"{len(adas_same_day_excluded):,}"
)

print(
    "Final clean longitudinal ADAS rows: "
    f"{len(adas_clean_longitudinal):,}"
)

print(
    "Total records in ADAS exclusion log: "
    f"{len(adas_excluded):,}"
)

# Confirm that no repeated participant-date combinations remain.
remaining_same_day_duplicates = int(
    adas_clean_longitudinal
    .duplicated(
        subset=["RID", "VISDATE"],
        keep=False
    )
    .sum()
)

print(
    "Repeated RID + VISDATE rows remaining: "
    f"{remaining_same_day_duplicates:,}"
)

## 1.19. Inspect baseline ADAS records and participant re-enrolment

examine the visit-code distribution in the cleaned longitudinal ADAS table and identify records labelled as baseline.

also check whether any participant has ADAS records in more than one ADNI phase. This is important because a participant may have a separate baseline visit when re-enrolling in a later ADNI phase. The final thesis baseline must therefore be aligned to the baseline definition in the labelled master cohort rather than selected from the ADAS table alone.

In [ ]:
# Summarise the most frequent translated visit codes.
visit_code_summary = (
    adas_clean_longitudinal["VISCODE2"]
    .value_counts(dropna=False)
    .rename_axis("VISCODE2")
    .reset_index(name="assessment_rows")
)

visit_code_summary["percent_of_records"] = (
    visit_code_summary["assessment_rows"]
    .div(len(adas_clean_longitudinal))
    .mul(100)
    .round(2)
)

print("Most frequent ADAS visit codes:")
display(visit_code_summary.head(30))

# Identify records explicitly labelled as baseline.
adas_labelled_baseline = (
    adas_clean_longitudinal.loc[
        adas_clean_longitudinal["VISCODE2"]
        .astype("string")
        .str.lower()
        .eq("bl")
    ]
    .copy()
)

# Count the number of ADNI phases represented for each participant.
participant_phase_counts = (
    adas_clean_longitudinal
    .groupby("RID")["PHASE"]
    .nunique()
)

multi_phase_rids = participant_phase_counts[
    participant_phase_counts > 1
].index

baseline_summary = pd.DataFrame(
    {
        "measure": [
            "Clean longitudinal assessment rows",
            "Unique participants with any clean ADAS record",
            "Rows labelled as baseline",
            "Unique participants with a labelled baseline",
            "Participants represented in multiple ADNI phases",
        ],
        "count": [
            len(adas_clean_longitudinal),
            adas_clean_longitudinal["RID"].nunique(),
            len(adas_labelled_baseline),
            adas_labelled_baseline["RID"].nunique(),
            len(multi_phase_rids),
        ],
    }
)

print("\nBaseline and phase summary:")
display(baseline_summary)

# Check whether any participant has more than one baseline-labelled ADAS record.
multiple_baseline_records = (
    adas_labelled_baseline
    .groupby("RID")
    .size()
    .loc[lambda counts: counts > 1]
    .sort_values(ascending=False)
)

print(
    "\nParticipants with more than one baseline-labelled ADAS record: "
    f"{len(multiple_baseline_records):,}"
)

if len(multiple_baseline_records) > 0:
    display(
        adas_labelled_baseline.loc[
            adas_labelled_baseline["RID"].isin(
                multiple_baseline_records.index
            ),
            [
                "PHASE",
                "PTID",
                "RID",
                "VISCODE",
                "VISCODE2",
                "VISDATE",
                "TOTSCORE",
                "TOTAL13",
            ],
        ]
        .sort_values(["RID", "VISDATE", "PHASE"])
        .reset_index(drop=True)
    )

## 1.20. Overview of ADAS visit and baseline coverage

The final cleaned longitudinal ADAS dataset contains 12,877 valid assessments from 2,960 unique participants. These records have passed the current table-level quality-control checks, including assessment completion, total-score availability, valid score ranges, participant identification, visit-date availability, unresolved quality-control errors, and ambiguous same-day records.

A total of 2,948 participants have exactly one ADAS record explicitly labelled as baseline through `VISCODE2 = "bl"`. No participant has more than one baseline-labelled ADAS record. The remaining 12 participants have valid longitudinal ADAS data but no assessment explicitly labelled as baseline.

The dataset also contains substantial longitudinal coverage across months 6, 12, 24, 36, 48, and later visits. These follow-up assessments will be preserved rather than discarded because the final reference-date and cross-modality alignment rules have not yet been established.

A total of 1,031 participants have ADAS records across more than one ADNI phase. These records will remain unchanged at this stage. Cross-phase re-enrolment and the choice of a study-wide reference date will be addressed only after all relevant clinical, demographic, cognitive, biomarker, genetic, and visit-date tables have been reviewed.

The ADAS table is therefore being processed as an independent modality. It will not yet be restricted to the preliminary DXSUM clinical reference cohort. The cleaned longitudinal table, the explicitly labelled baseline subset, and the exclusion log will be saved separately so that they can later support cross-modality coverage analysis and construction of the final participant-level cohort.

## 1.21. Save the interim cleaned ADAS outputs

save the independently cleaned ADAS source table without selecting one record per participant or merging diagnosis labels.

Three outputs will be created:

- the complete cleaned longitudinal ADAS table;
- the exclusion log with explicit reasons for each removed record;
- a compact quality-control summary describing the retained and excluded data.

These files are interim modality-specific outputs. They will later be revisited together with the other cleaned modalities when the common reference-date and participant-level cohort rules are defined.

In [ ]:
from pathlib import Path
import pandas as pd

# Define the ADAS output directories.
non_imaging_root = Path(
    "/content/drive/MyDrive/adni_mri/adni_non_imaging"
)

processed_dir = non_imaging_root / "processed"
qc_dir = non_imaging_root / "qc"

processed_dir.mkdir(parents=True, exist_ok=True)
qc_dir.mkdir(parents=True, exist_ok=True)

# Create readable exclusion reasons for the original QC exclusions.
flag_reason_map = {
    "flag_explicitly_not_done": "Assessment explicitly marked as not completed",
    "flag_unresolved_qc_error": "Unresolved ADNI quality-control error",
    "flag_missing_both_totals": "Both TOTSCORE and TOTAL13 are missing",
    "flag_totscore_out_of_range": "TOTSCORE outside the valid 0–70 range",
    "flag_total13_out_of_range": "TOTAL13 outside the valid 0–85 range",
    "flag_missing_rid": "Missing participant RID",
    "flag_missing_visit_date": "Missing visit date",
}

def build_exclusion_reason(row):
    """Combine all applicable row-level exclusion reasons."""
    existing_reason = row.get("exclusion_reason")

    if pd.notna(existing_reason) and str(existing_reason).strip():
        return str(existing_reason).strip()

    reasons = [
        reason
        for flag, reason in flag_reason_map.items()
        if flag in row.index and row[flag] is True
    ]

    return "; ".join(reasons) if reasons else "Excluded during ADAS quality control"

adas_excluded["exclusion_reason"] = adas_excluded.apply(
    build_exclusion_reason,
    axis=1
)

# Define output paths.
clean_path = (
    processed_dir
    / "adas_clean_longitudinal_interim.csv"
)

excluded_path = (
    qc_dir
    / "adas_exclusion_log.csv"
)

summary_path = (
    qc_dir
    / "adas_qc_summary.csv"
)

# Build a compact QC summary.
adas_qc_summary = pd.DataFrame(
    {
        "measure": [
            "Original ADAS rows",
            "Original unique participants",
            "Clean longitudinal rows retained",
            "Clean unique participants retained",
            "Excluded rows",
            "Retained percentage",
            "Participants with labelled baseline visit",
            "Participants without labelled baseline visit",
            "Participants represented in multiple ADNI phases",
            "Repeated RID + VISCODE2 combinations remaining",
            "Repeated RID + VISDATE combinations remaining",
        ],
        "value": [
            len(adas_raw),
            adas_raw["RID"].nunique(dropna=True),
            len(adas_clean_longitudinal),
            adas_clean_longitudinal["RID"].nunique(dropna=True),
            len(adas_excluded),
            round(
                len(adas_clean_longitudinal)
                / len(adas_raw)
                * 100,
                2
            ),
            adas_labelled_baseline["RID"].nunique(dropna=True),
            (
                adas_clean_longitudinal["RID"].nunique(dropna=True)
                - adas_labelled_baseline["RID"].nunique(dropna=True)
            ),
            len(multi_phase_rids),
            int(
                adas_clean_longitudinal
                .duplicated(
                    subset=["RID", "VISCODE2"],
                    keep=False
                )
                .sum()
            ),
            int(
                adas_clean_longitudinal
                .duplicated(
                    subset=["RID", "VISDATE"],
                    keep=False
                )
                .sum()
            ),
        ],
    }
)

# Save all interim ADAS outputs.
adas_clean_longitudinal.to_csv(
    clean_path,
    index=False
)

adas_excluded.to_csv(
    excluded_path,
    index=False
)

adas_qc_summary.to_csv(
    summary_path,
    index=False
)

print("ADAS interim outputs saved successfully.\n")

print(f"Clean longitudinal table:\n{clean_path}")
print(f"\nExclusion log:\n{excluded_path}")
print(f"\nQC summary:\n{summary_path}")

print("\nSaved output sizes:")
for path in [clean_path, excluded_path, summary_path]:
    print(f"- {path.name}: {path.stat().st_size / 1024:.2f} KB")

print("\nFinal QC summary:")
display(adas_qc_summary)

print("\nExclusion reasons:")
display(
    adas_excluded["exclusion_reason"]
    .value_counts(dropna=False)
    .rename_axis("exclusion_reason")
    .reset_index(name="records")
)